# Result analysis

This notebook generates the tables, figures and statistical summaries from the completed FST scan.

Main design:

- four 2022 female *Anopheles coluzzii* populations, all `n >= 50`;
- six population pairs;
- X and 3R;
- 0-fold and 4-fold coding sites;
- non-overlapping 1 Mb windows;
- Dataset A = frozen X:13,393,109–17,393,108 focal interval;
- Dataset B = all analysed windows, with only the four Dataset A X windows removed;
- windows with fewer than 1000 usable sites are flagged, not silently deleted;
- secondary peaks use robust within-comparison thresholds and are interpreted using absolute FST, recurrence, clustering and site-count QC.

## Setup

In [1]:
from google.colab import drive
drive.mount("/content/drive")

# Standard scientific Python packages used by the downstream analysis.
%pip install -q "pandas==2.2.2" scipy statsmodels matplotlib

from pathlib import Path
import os
import re
import math
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import wilcoxon, friedmanchisquare
from statsmodels.stats.multitest import multipletests

pd.set_option("display.max_columns", 160)
pd.set_option("display.max_rows", 250)

# This is the same folder used by final_fst_scan.ipynb.
OUTPUT_ROOT = Path("/content/drive/MyDrive/FYP/final_output")

DATA_DIR = OUTPUT_ROOT / "data"
TABLE_DIR = OUTPUT_ROOT / "tables"
FIG_DIR = OUTPUT_ROOT / "figures"

for folder in [DATA_DIR, TABLE_DIR, FIG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

FST_FILE = DATA_DIR / "FST_selected6pairs_1Mb_3R_X_full_scan_minN50.csv"
SITE_COUNT_FILE = DATA_DIR / "site_counts_1Mb_windows_3R_X_existing_counts.csv"
SAMPLE_FILE = DATA_DIR / "selected_samples_exact.csv"

if not FST_FILE.exists():
    raise FileNotFoundError(
        "Final FST file not found. Run final_fst_scan.ipynb until the scan is complete."
    )

print("Final output folder:", OUTPUT_ROOT)
print("FST file:", FST_FILE)

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 56.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 2.2.2 which is incompatible.


FileNotFoundError: Final FST file not found. Run final_fst_scan.ipynb until the scan is complete.

## Load and validate the completed FST scan

In [ ]:
fst = pd.read_csv(FST_FILE)

# Define the set of required column names
required = {
    "Pair_ID", "Pair_label", "Population_1", "Population_2",
    "Contig", "Region", "Genomic_start_POS", "Genomic_end_POS",
    "Distance_from_CEN_start", "Distance_from_CEN_end",
    "Distance_from_CEN_midpoint_Mb", "Site_class",
    "FST_estimate", "Run_status"
}

# Raise an error if any required column is missing
missing = sorted(required - set(fst.columns))
if missing:
    raise KeyError(f"Missing required FST columns: {missing}")

fst["FST_estimate"] = pd.to_numeric(fst["FST_estimate"], errors="coerce")

# Filter for successful runs and valid site classes (0-fold & 4-fold)
use = fst[
    fst["Run_status"].eq("success")
    & fst["Site_class"].isin(["0-fold", "4-fold"])
].copy()

# Deduplicate based on key identifying features
use = use.drop_duplicates(
    ["Pair_ID", "Contig", "Region", "Site_class"]
)

print("Successful FST rows:", len(use))
print("Population pairs:", use["Pair_ID"].nunique())
print("Chromosomes:", sorted(use["Contig"].unique()))
print("Site classes:", sorted(use["Site_class"].unique()))

## Create one paired row per pair × chromosome × window

In [ ]:
# Identify metadata columns present in the dataframe to preserve during reshaping
meta_cols = [
    c for c in [
        "Comparison_group", "Pair_ID", "Pair_short", "Pair_label",
        "Population_1", "Population_2", "Country_1", "Country_2",
        "Location_1", "Location_2", "N_population_1", "N_population_2",
        "Contig", "Region", "Genomic_start_POS", "Genomic_end_POS",
        "Distance_from_CEN_start", "Distance_from_CEN_end",
        "Distance_from_CEN_midpoint_Mb", "Window_size_bp"
    ]
    if c in use.columns
]

# Pivot table to widen 0-fold and 4-fold site classes into separate columns
fst_wide = (
    use.pivot_table(
        index=meta_cols,
        columns="Site_class",
        values="FST_estimate",
        aggfunc="first"
    )
    .reset_index()
    .rename(columns={"0-fold": "FST0", "4-fold": "FST4"})
)

# Ensure numeric data types for both FST estimates
fst_wide["FST0"] = pd.to_numeric(fst_wide["FST0"], errors="coerce")
fst_wide["FST4"] = pd.to_numeric(fst_wide["FST4"], errors="coerce")
fst_wide["Delta_FST0_minus_FST4"] = fst_wide["FST0"] - fst_wide["FST4"]

print("Paired pair × chromosome × window rows:", len(fst_wide))
display(fst_wide.head())

## Merge verified usable-site counts

In [ ]:
# Load window site-count metadata
site_counts = pd.read_csv(SITE_COUNT_FILE)

required_site = {"Contig", "Region", "N_sites_FST0", "N_sites_FST4"}
missing_site = sorted(required_site - set(site_counts.columns))
if missing_site:
    raise KeyError(f"Missing site-count columns: {missing_site}")

# Deduplicate and rename site-count columns
site_counts = (
    site_counts[
        ["Contig", "Region", "N_sites_FST0", "N_sites_FST4"]
    ]
    .drop_duplicates(["Contig", "Region"])
    .rename(columns={
        "N_sites_FST0": "N_sites_0fold",
        "N_sites_FST4": "N_sites_4fold",
    })
)

# Merge usable site counts into the main dataset
fst_wide = fst_wide.merge(
    site_counts,
    on=["Contig", "Region"],
    how="left",
    validate="many_to_one"
)

MIN_USABLE_SITES_QC = 1000

# Flag windows below the site-count threshold
fst_wide["Low_sites_0fold"] = (
    pd.to_numeric(fst_wide["N_sites_0fold"], errors="coerce")
    < MIN_USABLE_SITES_QC
)
fst_wide["Low_sites_4fold"] = (
    pd.to_numeric(fst_wide["N_sites_4fold"], errors="coerce")
    < MIN_USABLE_SITES_QC
)

print("Rows with missing 0-fold site counts:", fst_wide["N_sites_0fold"].isna().sum())
print("Rows with missing 4-fold site counts:", fst_wide["N_sites_4fold"].isna().sum())
print("Rows flagged <1000 at 0-fold:", fst_wide["Low_sites_0fold"].sum())
print("Rows flagged <1000 at 4-fold:", fst_wide["Low_sites_4fold"].sum())

## Classify Nigeria-containing and non-Nigeria pairs

In [ ]:
# Target population for subgrouping
NIGERIA_POPULATION = "Nigeria_Gombe"

# Helper function to check if a row involves the target Nigeria population
def involves_population(row, population=NIGERIA_POPULATION):
    p1 = str(row.get("Population_1", "")).strip()
    p2 = str(row.get("Population_2", "")).strip()
    pair_id = str(row.get("Pair_ID", "")).strip()
    pair_label = str(row.get("Pair_label", "")).strip()

    return (
        p1 == population
        or p2 == population
        or population in pair_id
        or "NG_Gombe" in pair_label
    )

# Assign classification labels based on Nigeria presence
fst_wide["Nigeria_status"] = np.where(
    fst_wide.apply(involves_population, axis=1),
    "With Nigeria_Gombe",
    "Without Nigeria"
)

pair_status = (
    fst_wide[["Pair_ID", "Pair_label", "Nigeria_status"]]
    .drop_duplicates()
    .sort_values(["Nigeria_status", "Pair_label"])
)

display(pair_status)

# Validate that pair counts split evenly (3 vs 3) as expected
status_counts = (
    pair_status.groupby("Nigeria_status")["Pair_ID"].nunique().to_dict()
)

if status_counts != {
    "With Nigeria_Gombe": 3,
    "Without Nigeria": 3
}:
    raise ValueError(f"Nigeria classification failed: {status_counts}")

## Freeze Dataset A and construct Dataset B

Dataset A is the four-window recurrent Nigeria-associated X interval:

- X:13,393,109–17,393,108
- strongest descriptive core: X:14,393,109–16,393,108

The interval is fixed by genomic coordinates and the exact same four windows are used in all six pairs.

Dataset B removes these four windows **only from X**. No 3R windows are excluded.

In [ ]:
# Define genomic coordinates for focal region Dataset A
DATASET_A_CONTIG = "X"
DATASET_A_START = 13_393_109
DATASET_A_END = 17_393_108

CORE_START = 14_393_109
CORE_END = 16_393_108

# Extract window regions that belong to Dataset A
dataset_A_regions_df = (
    fst_wide[
        fst_wide["Contig"].eq(DATASET_A_CONTIG)
        & (fst_wide["Genomic_start_POS"] >= DATASET_A_START)
        & (fst_wide["Genomic_end_POS"] <= DATASET_A_END)
    ][[
        "Region", "Genomic_start_POS", "Genomic_end_POS",
        "Distance_from_CEN_start", "Distance_from_CEN_end",
        "Distance_from_CEN_midpoint_Mb"
    ]]
    .drop_duplicates("Region")
    .sort_values("Genomic_start_POS")
    .reset_index(drop=True)
)

if len(dataset_A_regions_df) != 4:
    raise ValueError(
        f"Expected 4 Dataset A windows, found {len(dataset_A_regions_df)}"
    )

dataset_A_regions = set(dataset_A_regions_df["Region"].astype(str))

dataset_A = fst_wide[
    fst_wide["Contig"].eq("X")
    & fst_wide["Region"].astype(str).isin(dataset_A_regions)
].copy()

dataset_B = fst_wide[
    ~(
        fst_wide["Contig"].eq("X")
        & fst_wide["Region"].astype(str).isin(dataset_A_regions)
    )
].copy()

print("Dataset A windows:")
display(dataset_A_regions_df)

print("Dataset A rows:", len(dataset_A))
print("Dataset B rows:", len(dataset_B))
print(
    "Dataset B windows by chromosome:",
    dataset_B.groupby("Contig")["Region"].nunique().to_dict()
)

## Full X and 3R scans across all six population pairs

In [ ]:
# Sort population pairs so Nigeria-containing pairs are plotted first
pair_order = (
    fst_wide[["Pair_ID", "Pair_label", "Nigeria_status"]]
    .drop_duplicates()
    .assign(
        status_order=lambda d: np.where(
            d["Nigeria_status"].eq("With Nigeria_Gombe"), 0, 1
        )
    )
    .sort_values(["status_order", "Pair_label"])
)

# Generate 2x3 subplot grids for full genome scans per chromosome
for contig in ["X", "3R"]:
    fig, axes = plt.subplots(
        2, 3, figsize=(13, 7), sharex=True, sharey=True
    )

    for ax, (_, pair) in zip(axes.ravel(), pair_order.iterrows()):
        p = fst_wide[
            fst_wide["Pair_ID"].eq(pair["Pair_ID"])
            & fst_wide["Contig"].eq(contig)
        ].sort_values("Distance_from_CEN_midpoint_Mb")
        # Plot 0-fold FST curve
        ax.plot(
            p["Distance_from_CEN_midpoint_Mb"],
            p["FST0"],
            color="black",
            marker="o",
            markersize=3,
            linewidth=1.2,
            label="0-fold"
        )

        ax.plot(
            p["Distance_from_CEN_midpoint_Mb"],
            p["FST4"],
            color="red",
            marker="s",
            markersize=2.8,
            linewidth=1.2,
            label="4-fold"
        )

        ax.set_title(pair["Pair_label"], fontsize=9)
        ax.set_xlabel("Distance from centromere (Mb)")
        ax.set_ylabel("FST")
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=2, frameon=False)
    fig.tight_layout(rect=[0, 0, 1, 0.95])

    fig.savefig(
        FIG_DIR / f"Figure_full_{contig}_six_pairs.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()

## Dataset B background means and SE

In [ ]:
# Melt Dataset B to long format for aggregated calculations
b_long = dataset_B.melt(
    id_vars=[
        c for c in [
            "Pair_ID", "Pair_label", "Nigeria_status",
            "Contig", "Region"
        ]
        if c in dataset_B.columns
    ],
    value_vars=["FST0", "FST4"],
    var_name="Site_class",
    value_name="FST"
)

b_long["Site_class"] = b_long["Site_class"].map({
    "FST0": "0-fold",
    "FST4": "4-fold"
})
b_long["FST"] = pd.to_numeric(b_long["FST"], errors="coerce")

# Calculate background summary statistics (Mean, SD, N)
background_table = (
    b_long
    .groupby(
        ["Nigeria_status", "Pair_ID", "Pair_label", "Contig", "Site_class"],
        dropna=False
    )["FST"]
    .agg(
        Mean_FST="mean",
        SD_FST="std",
        N_windows="count"
    )
    .reset_index()
)

# Compute Standard Error of the Mean (SE)
background_table["SE_FST"] = np.where(
    background_table["N_windows"] > 1,
    background_table["SD_FST"] / np.sqrt(background_table["N_windows"]),
    np.nan
)

background_table.to_csv(
    TABLE_DIR / "TableB1_background_FST_by_pair_chr_site.csv",
    index=False
)

display(background_table)

## B-H1: exploratory population-pair effect with matched windows

In [ ]:
# Friedman tests compare all six population pairs while keeping genomic windows matched within each chromosome and site class.
friedman_rows = []
posthoc_rows = []

for contig in ["X", "3R"]:
    for site_class, value_col in [
        ("0-fold", "FST0"),
        ("4-fold", "FST4"),
    ]:
        sub = dataset_B[dataset_B["Contig"].eq(contig)].copy()

        wide = sub.pivot(
            index="Region",
            columns="Pair_ID",
            values=value_col
        ).dropna()

        pair_ids = wide.columns.tolist()

        # Perform non-parametric Friedman test across matched windows
        stat, pval = friedmanchisquare(
            *[wide[pair_id].to_numpy() for pair_id in pair_ids]
        )

        friedman_rows.append({
            "Contig": contig,
            "Site_class": site_class,
            "N_matched_windows": len(wide),
            "N_pairs": len(pair_ids),
            "Friedman_chi2": stat,
            "Raw_p": pval,
        })

        for i in range(len(pair_ids)):
            for j in range(i + 1, len(pair_ids)):
                a, b = pair_ids[i], pair_ids[j]
                w_stat, w_p = wilcoxon(
                    wide[a],
                    wide[b],
                    alternative="two-sided",
                    zero_method="wilcox"
                )

                posthoc_rows.append({
                    "Contig": contig,
                    "Site_class": site_class,
                    "Pair_1": a,
                    "Pair_2": b,
                    "N_matched_windows": len(wide),
                    "Mean_difference": (wide[a] - wide[b]).mean(),
                    "Wilcoxon_statistic": w_stat,
                    "Raw_p": w_p,
                })

friedman_tests = pd.DataFrame(friedman_rows)
friedman_tests["FDR_p"] = multipletests(
    friedman_tests["Raw_p"],
    method="fdr_bh"
)[1]

posthoc_tests = pd.DataFrame(posthoc_rows)
posthoc_tests["FDR_p"] = multipletests(
    posthoc_tests["Raw_p"],
    method="fdr_bh"
)[1]

friedman_tests.to_csv(
    TABLE_DIR / "TableB0_population_pair_effect_friedman.csv",
    index=False
)
posthoc_tests.to_csv(
    TABLE_DIR / "TableB0_population_pair_posthoc_wilcoxon.csv",
    index=False
)

display(friedman_tests)

## B-H2 and B-H3: paired background tests

The six pairwise comparisons are the paired units.

- B-H2: X versus 3R, separately for 0-fold and 4-fold.
- B-H3: 0-fold versus 4-fold, separately for X and 3R.

Because the six comparisons share populations, p-values are interpreted cautiously and effect direction is reported alongside FDR-adjusted p-values.

In [ ]:
# Pair-level means are the paired units for chromosome and site-class tests.
# Because the six pairwise comparisons share populations, these p-values are interpreted cautiously and are reported with effect direction.
pair_means_B = (
    b_long.groupby(
        ["Pair_ID", "Pair_label", "Nigeria_status", "Contig", "Site_class"],
        as_index=False
    )
    .agg(Mean_FST=("FST", "mean"))
)

tests = []

# B-H2: Test X vs 3R chromosome effect per site class
for site in ["0-fold", "4-fold"]:
    p = pair_means_B[
        pair_means_B["Site_class"].eq(site)
    ].pivot(
        index=["Pair_ID", "Pair_label"],
        columns="Contig",
        values="Mean_FST"
    ).dropna(subset=["X", "3R"])

    diff = p["X"] - p["3R"]
    stat, pval = wilcoxon(
        p["X"], p["3R"],
        alternative="two-sided",
        zero_method="wilcox"
    )

    tests.append({
        "Hypothesis_ID": "B-H2",
        "Comparison": f"X vs 3R — {site}",
        "N_pairs": len(p),
        "Mean_difference": diff.mean(),
        "Median_difference": diff.median(),
        "Direction_count_positive": int((diff > 0).sum()),
        "Wilcoxon_statistic": stat,
        "Raw_p": pval,
    })

# B-H3: Test 0-fold vs 4-fold site-class effect per chromosome
for contig in ["X", "3R"]:
    p = pair_means_B[
        pair_means_B["Contig"].eq(contig)
    ].pivot(
        index=["Pair_ID", "Pair_label"],
        columns="Site_class",
        values="Mean_FST"
    ).dropna(subset=["0-fold", "4-fold"])

    diff = p["0-fold"] - p["4-fold"]
    stat, pval = wilcoxon(
        p["0-fold"], p["4-fold"],
        alternative="two-sided",
        zero_method="wilcox"
    )

    tests.append({
        "Hypothesis_ID": "B-H3",
        "Comparison": f"0-fold vs 4-fold — {contig}",
        "N_pairs": len(p),
        "Mean_difference": diff.mean(),
        "Median_difference": diff.median(),
        "Direction_count_positive": int((diff > 0).sum()),
        "Wilcoxon_statistic": stat,
        "Raw_p": pval,
    })

# Adjust raw p-values using FDR
paired_tests = pd.DataFrame(tests)
paired_tests["FDR_p"] = multipletests(
    paired_tests["Raw_p"],
    method="fdr_bh"
)[1]

paired_tests.to_csv(
    TABLE_DIR / "TableB3_paired_background_tests.csv",
    index=False
)

display(paired_tests)

## Dataset B: Nigeria-containing versus non-Nigeria background

In [ ]:
# Summarize background FST between Nigeria-containing and non-Nigeria pairs
background_group_summary = (
    pair_means_B
    .groupby(["Nigeria_status", "Contig", "Site_class"], as_index=False)
    .agg(
        N_pairwise_comparisons=("Pair_ID", "nunique"),
        Mean_of_pair_means=("Mean_FST", "mean"),
        SD_of_pair_means=("Mean_FST", "std"),
        Min_pair_mean=("Mean_FST", "min"),
        Max_pair_mean=("Mean_FST", "max"),
    )
)
#Compute standard error across pair-level means
background_group_summary["SE_of_pair_means"] = (
    background_group_summary["SD_of_pair_means"]
    / np.sqrt(background_group_summary["N_pairwise_comparisons"])
)

background_group_summary.to_csv(
    TABLE_DIR / "TableB4_Nigeria_vs_nonNigeria_background.csv",
    index=False
)

display(background_group_summary)

## Background figures after focal-X exclusion

In [ ]:
# Determine uniform y-axis maximum across background figures
global_ymax = max(
    (
        background_table["Mean_FST"]
        + background_table["SE_FST"].fillna(0)
    ).max() * 1.18,
    0.001
)

# Render background bar plots for chromosomes X and 3R
for contig in ["X", "3R"]:
    p = background_table[
        background_table["Contig"].eq(contig)
    ].copy()

    p["Status_order"] = np.where(
        p["Nigeria_status"].eq("With Nigeria_Gombe"), 0, 1
    )
    # Establish pair ordering
    po = (
        p[["Pair_ID", "Pair_label", "Nigeria_status", "Status_order"]]
        .drop_duplicates()
        .sort_values(["Status_order", "Pair_label"])
    )

    pair_ids = po["Pair_ID"].tolist()
    labels = po["Pair_label"].tolist()

    f0 = (
        p[p["Site_class"].eq("0-fold")]
        .set_index("Pair_ID")
        .reindex(pair_ids)
    )
    f4 = (
        p[p["Site_class"].eq("4-fold")]
        .set_index("Pair_ID")
        .reindex(pair_ids)
    )

    x = np.arange(len(pair_ids))
    w = 0.38

    fig, ax = plt.subplots(figsize=(11.5, 5.9))
    # Plot paired bars with error indicators (SE)
    ax.bar(
        x - w / 2,
        f0["Mean_FST"],
        w,
        yerr=f0["SE_FST"],
        capsize=3,
        color="black",
        edgecolor="black",
        label="0-fold"
    )

    ax.bar(
        x + w / 2,
        f4["Mean_FST"],
        w,
        yerr=f4["SE_FST"],
        capsize=3,
        color="red",
        edgecolor="red",
        label="4-fold"
    )

    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=35, ha="right")
    ax.set_ylabel("Background mean FST ± SE")
    ax.set_title(f"Peak-excluded background differentiation — {contig}")
    ax.set_ylim(0, global_ymax)
    ax.legend(frameon=False)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    fig.tight_layout()
    fig.savefig(
        FIG_DIR / f"Figure_background_{contig}.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()

## Dataset A: exact focal interval across all six pairs

In [ ]:
# Compute overall mean focal FST within Dataset A per population pair
peak_all_six = (
    dataset_A
    .groupby(["Nigeria_status", "Pair_ID", "Pair_label"], as_index=False)
    .agg(
        N_peak_windows=("Region", "nunique"),
        Mean_peak_FST0=("FST0", "mean"),
        SE_peak_FST0=("FST0", "sem"),
        Mean_peak_FST4=("FST4", "mean"),
        SE_peak_FST4=("FST4", "sem"),
        Max_peak_FST0=("FST0", "max"),
        Max_peak_FST4=("FST4", "max"),
    )
)

peak_all_six["Mean_peak_delta_FST0_minus_FST4"] = (
    peak_all_six["Mean_peak_FST0"]
    - peak_all_six["Mean_peak_FST4"]
)

peak_all_six.to_csv(
    TABLE_DIR / "TableA2_peak_all_six_pairs.csv",
    index=False
)

display(peak_all_six)

## Dataset A: four-window site-class pattern across Nigeria comparisons

In [ ]:
ng_A = dataset_A[
    dataset_A["Nigeria_status"].eq("With Nigeria_Gombe")
].copy()
# Summarise the four focal X windows across the three Nigeria comparisons.
# For each window, calculate the mean and standard error of FST
# separately for 0-fold and 4-fold sites.
peak_siteclass_window = (
    ng_A
    .groupby([
        "Region", "Genomic_start_POS", "Genomic_end_POS",
        "Distance_from_CEN_midpoint_Mb"
    ], as_index=False)
    .agg(
        N_Nigeria_pairs=("Pair_ID", "nunique"),
        Mean_FST0=("FST0", "mean"),
        SE_FST0=("FST0", "sem"),
        Mean_FST4=("FST4", "mean"),
        SE_FST4=("FST4", "sem"),
    )
    .sort_values("Genomic_start_POS")
)

# Calculate the difference between 0-fold and 4-fold mean FST
peak_siteclass_window["Delta_FST0_minus_FST4"] = (
    peak_siteclass_window["Mean_FST0"]
    - peak_siteclass_window["Mean_FST4"]
)

peak_siteclass_window.to_csv(
    TABLE_DIR / "TableA3_peak_siteclass_by_window.csv",
    index=False
)

display(peak_siteclass_window)

## Dataset A: exploratory pair-level 0-fold versus 4-fold test

In [ ]:
# Calculate the mean focal-region FST for each Nigeria-containing pair.
# This gives one 0-fold value and one 4-fold value per population pair.
ah2_pair_means = (
    ng_A
    .groupby(["Pair_ID", "Pair_label"], as_index=False)
    .agg(
        Mean_peak_FST0=("FST0", "mean"),
        Mean_peak_FST4=("FST4", "mean"),
    )
)

ah2_pair_means["Delta_FST0_minus_FST4"] = (
    ah2_pair_means["Mean_peak_FST0"]
    - ah2_pair_means["Mean_peak_FST4"]
)
# Use a paired Wilcoxon signed-rank test to compare
ah2_stat, ah2_p = wilcoxon(
    ah2_pair_means["Mean_peak_FST0"],
    ah2_pair_means["Mean_peak_FST4"],
    alternative="two-sided",
    zero_method="wilcox"
)

# Store the test result in a small summary table.
ah2_test = pd.DataFrame([{
    "Hypothesis_ID": "A-H2",
    "N_Nigeria_pairs": len(ah2_pair_means),
    "Mean_delta_FST0_minus_FST4":
        ah2_pair_means["Delta_FST0_minus_FST4"].mean(),
    "Wilcoxon_statistic": ah2_stat,
    "Raw_p": ah2_p,
    "Interpretation_limit":
        "Exploratory only: n=3 pair-level observations."
}])

ah2_pair_means.to_csv(
    TABLE_DIR / "TableA3b_peak_pair_level_siteclass_means.csv",
    index=False
)
ah2_test.to_csv(
    TABLE_DIR / "TableA3c_peak_pair_level_wilcoxon.csv",
    index=False
)

display(ah2_pair_means)
display(ah2_test)

## Dataset A figures

In [ ]:
# Get the three Nigeria-containing population pairs.
ng_pairs = (
    ng_A[["Pair_ID", "Pair_label"]]
    .drop_duplicates()
    .sort_values("Pair_label")
)

# Full X profile for the three Nigeria comparisons.
linestyles = ["-", "--", ":"]

fig, ax = plt.subplots(figsize=(12, 6.5))

for ls, (_, pair) in zip(linestyles, ng_pairs.iterrows()):
    p = fst_wide[
        fst_wide["Pair_ID"].eq(pair["Pair_ID"])
        & fst_wide["Contig"].eq("X")
    ].sort_values("Distance_from_CEN_midpoint_Mb")
    # Plot 0-fold FST in black.
    ax.plot(
        p["Distance_from_CEN_midpoint_Mb"],
        p["FST0"],
        color="black",
        linestyle=ls,
        marker="o",
        markersize=3,
        linewidth=1.5,
        label=f"{pair['Pair_label']} — 0-fold"
    )
    # Plot 4-fold FST in red.
    ax.plot(
        p["Distance_from_CEN_midpoint_Mb"],
        p["FST4"],
        color="red",
        linestyle=ls,
        marker="s",
        markersize=2.5,
        linewidth=1.2,
        label=f"{pair['Pair_label']} — 4-fold"
    )

# Shade the four-window recurrent Nigeria-associated X region.
ax.axvspan(
    dataset_A_regions_df["Distance_from_CEN_start"].min() / 1e6,
    dataset_A_regions_df["Distance_from_CEN_end"].max() / 1e6,
    color="grey",
    alpha=0.15,
    label="Recurrent Nigeria-associated X peak"
)

ax.set_xlabel("Distance from X centromere (Mb)")
ax.set_ylabel("FST")
ax.set_title("Full X profile: Nigeria-containing comparisons")
ax.legend(fontsize=7.5, ncol=2, frameon=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

fig.tight_layout()
fig.savefig(
    FIG_DIR / "FigureA0_full_X_profile_Nigeria_pairs.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

# Four-window mean site-class pattern.
q = peak_siteclass_window.copy()
x = np.arange(len(q))
w = 0.38

fig, ax = plt.subplots(figsize=(9, 5.5))

ax.bar(
    x - w / 2,
    q["Mean_FST0"],
    width=w,
    yerr=q["SE_FST0"],
    capsize=4,
    color="black",
    edgecolor="black",
    label="0-fold"
)

ax.bar(
    x + w / 2,
    q["Mean_FST4"],
    width=w,
    yerr=q["SE_FST4"],
    capsize=4,
    color="red",
    edgecolor="red",
    label="4-fold"
)

ax.set_xticks(x)
ax.set_xticklabels(
    [f"{v:.1f}" for v in q["Distance_from_CEN_midpoint_Mb"]]
)
ax.set_xlabel("Distance from X centromere (Mb; window midpoint)")
ax.set_ylabel("Mean FST across 3 Nigeria comparisons ± SE")
ax.set_title("Site-class pattern across the recurrent X peak")
ax.legend(frameon=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

fig.tight_layout()
fig.savefig(
    FIG_DIR / "FigureA3_peak_window_mean_FST0_FST4.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

## Site-count QC

In [ ]:
# Select the columns needed for site-count quality control.
qc_cols = [
    "Pair_ID", "Pair_label", "Nigeria_status", "Contig", "Region",
    "N_sites_0fold", "N_sites_4fold",
    "Low_sites_0fold", "Low_sites_4fold"
]

low_site_qc = fst_wide[qc_cols].copy()
# Save the full window-level QC table.
low_site_qc.to_csv(
    TABLE_DIR / "TableQC_all_windows_site_counts.csv",
    index=False
)
# Summarise how many rows have fewer than 1000 usable sites or are missing site-count information.
qc_summary = pd.DataFrame([{
    "Total_pair_window_rows": len(fst_wide),
    "Rows_lt1000_0fold": int(fst_wide["Low_sites_0fold"].sum()),
    "Rows_lt1000_4fold": int(fst_wide["Low_sites_4fold"].sum()),
    "Rows_missing_0fold_site_count": int(fst_wide["N_sites_0fold"].isna().sum()),
    "Rows_missing_4fold_site_count": int(fst_wide["N_sites_4fold"].isna().sum()),
}])

qc_summary.to_csv(
    TABLE_DIR / "TableQC_site_count_summary.csv",
    index=False
)

display(qc_summary)

## Exploratory isolation by distance

This reproduces the final notebook's exploratory IBD analysis using exact coordinates from the archived selected-sample metadata. It is treated as exploratory because six pairwise observations are derived from only four populations.

In [ ]:
if SAMPLE_FILE.exists():
    # Load the exact samples
    selected_samples_exact = pd.read_csv(SAMPLE_FILE)

    coord_rows = []

    for pop_id, q in selected_samples_exact.groupby("Population_ID"):
        qq = q.dropna(subset=["latitude", "longitude"]).copy()

        if qq.empty:
            continue
        # Count how many distinct sampling coordinates occur within each population.
        unique_coords = (
            qq[["latitude", "longitude"]]
            .value_counts()
            .reset_index(name="N_samples_at_coordinate")
        )

        # Use the average latitude and longitude as the population coordinate.
        coord_rows.append({
            "Population_ID": pop_id,
            "Latitude": np.average(qq["latitude"].astype(float)),
            "Longitude": np.average(qq["longitude"].astype(float)),
            "N_samples_with_coordinates": len(qq),
            "N_unique_coordinates": len(unique_coords),
        })

    population_coordinates = pd.DataFrame(coord_rows)

    # Calculate great-circle distance between two coordinates in kilometres.
    def haversine_km(lat1, lon1, lat2, lon2):
        R = 6371.0088
        phi1, phi2 = np.radians([lat1, lat2])
        dphi = np.radians(lat2 - lat1)
        dlambda = np.radians(lon2 - lon1)
        a = (
            np.sin(dphi / 2) ** 2
            + np.cos(phi1) * np.cos(phi2)
            * np.sin(dlambda / 2) ** 2
        )
        return 2 * R * np.arcsin(np.sqrt(a))

    pair_pop = (
        dataset_B[["Pair_ID", "Pair_label"]]
        .drop_duplicates("Pair_ID")
        .copy()
    )
    # Get the six population pairs used in Dataset B.
    pair_pop[["Population_1", "Population_2"]] = (
        pair_pop["Pair_ID"]
        .apply(lambda x: pd.Series(str(x).split("_vs_", 1)))
    )
    # Create a lookup table for population coordinates.
    coord_lookup = population_coordinates.set_index("Population_ID")

    distance_rows = []

    for _, row in pair_pop.iterrows():
        p1 = row["Population_1"]
        p2 = row["Population_2"]

        c1 = coord_lookup.loc[p1]
        c2 = coord_lookup.loc[p2]

        distance_rows.append({
            "Pair_ID": row["Pair_ID"],
            "Pair_label": row["Pair_label"],
            "Population_1": p1,
            "Population_2": p2,
            "Distance_km": haversine_km(
                c1["Latitude"], c1["Longitude"],
                c2["Latitude"], c2["Longitude"]
            ),
        })

    pair_distances = pd.DataFrame(distance_rows)

    # Combine geographic distance with the background mean FST
    ibd_data = pair_means_B.merge(
        pair_distances,
        on=["Pair_ID", "Pair_label"],
        how="left"
    )

    ibd_rows = []

    # Test whether geographic distance is associated with mean FST.
    for (contig, site), q in ibd_data.groupby(
        ["Contig", "Site_class"]
    ):
        q = q.dropna(subset=["Distance_km", "Mean_FST"])

        rho, p = stats.spearmanr(
            q["Distance_km"],
            q["Mean_FST"]
        )

        ibd_rows.append({
            "Contig": contig,
            "Site_class": site,
            "N_pairwise_points": len(q),
            "Spearman_rho": rho,
            "Raw_p": p,
        })

    ibd_tests = pd.DataFrame(ibd_rows)

    population_coordinates.to_csv(
        TABLE_DIR / "TableB5_IBD_population_coordinates.csv",
        index=False
    )
    pair_distances.to_csv(
        TABLE_DIR / "TableB5_IBD_pairwise_distances.csv",
        index=False
    )
    ibd_tests.to_csv(
        TABLE_DIR / "TableB5b_IBD_tests.csv",
        index=False
    )

    display(ibd_tests)

else:
    print("selected_samples_exact.csv not found; IBD section skipped.")

# Secondary differentiation landscape

Dataset A remains the primary focal interval and is excluded from all analyses below.

For each pair × chromosome × site class, a robust threshold is calculated as:

`median FST + 3 × 1.4826 × MAD`

A secondary signal is then described using:

- absolute FST;
- FST excess above its local median;
- recurrence across comparisons;
- adjacent-window clustering;
- usable-site-count QC.

`robust_z` is retained as a measure of unusualness, but is not used alone to rank biological importance.

In [ ]:
# Convert Dataset B to one row per pair × chromosome × window × site class.
secondary_parts = []

for site_class, fst_col, site_col in [
    ("0-fold", "FST0", "N_sites_0fold"),
    ("4-fold", "FST4", "N_sites_4fold"),
]:
    temp = dataset_B[[
        "Pair_ID", "Pair_label", "Nigeria_status",
        "Contig", "Region",
        "Genomic_start_POS", "Genomic_end_POS",
        "Distance_from_CEN_midpoint_Mb",
        fst_col, site_col
    ]].copy()

    temp = temp.rename(columns={
        fst_col: "FST",
        site_col: "N_sites",
    })
    temp["Site_class"] = site_class

    secondary_parts.append(temp)

secondary_long = pd.concat(
    secondary_parts,
    ignore_index=True
)

secondary_long["FST"] = pd.to_numeric(
    secondary_long["FST"],
    errors="coerce"
)
secondary_long["N_sites"] = pd.to_numeric(
    secondary_long["N_sites"],
    errors="coerce"
)

print("Secondary observations:", len(secondary_long))

In [ ]:
# Secondary peaks are defined relative to each pair/chromosome/site-class background using median + 3 scaled MAD. The focal X region is excluded.
def robust_background_stats(values):
    # Convert FST values to numeric and remove missing values.
    x = pd.to_numeric(
        pd.Series(values),
        errors="coerce"
    ).dropna()

    median = x.median()
    mad = np.median(np.abs(x - median))
    # Scale MAD to make it comparable to a standard deviation
    scale = 1.4826 * mad
    # If MAD is zero or unavailable, use a fallback robust scale estimate.
    if scale == 0 or np.isnan(scale):
        q1, q3 = x.quantile([0.25, 0.75])
        scale = (
            (q3 - q1) / 1.349
            if q3 > q1
            else x.std(ddof=1)
        )

    threshold = median + 3 * scale

    return pd.Series({
        "Background_median_FST": median,
        "MAD_scale": scale,
        "Robust_threshold": threshold,
    })
# Calculate the local background threshold separately for every pair * chromosome * site-class combination.
thresholds = (
    secondary_long
    .groupby(
        ["Pair_ID", "Pair_label", "Nigeria_status",
         "Contig", "Site_class"]
    )["FST"]
    .apply(robust_background_stats)
    .unstack()
    .reset_index()
)

secondary_windows = secondary_long.merge(
    thresholds,
    on=[
        "Pair_ID", "Pair_label", "Nigeria_status",
        "Contig", "Site_class"
    ],
    how="left"
)
# Flag windows whose FST is above the robust local threshold.
secondary_windows["Robust_outlier"] = (
    secondary_windows["FST"]
    > secondary_windows["Robust_threshold"]
)
# Calculate the absolute increase in FST above the local median.
secondary_windows["FST_excess_over_median"] = (
    secondary_windows["FST"]
    - secondary_windows["Background_median_FST"]
)

secondary_windows["robust_z"] = (
    secondary_windows["FST_excess_over_median"]
    / secondary_windows["MAD_scale"].replace(0, np.nan)
)

secondary_windows["Low_site_count"] = (
    secondary_windows["N_sites"] < MIN_USABLE_SITES_QC
)

secondary_windows.to_csv(
    TABLE_DIR / "Secondary_all_background_windows_v3_6.csv",
    index=False
)

print("Robust outlier observations:", int(secondary_windows["Robust_outlier"].sum()))

## Secondary recurrence by exact genomic window

In [ ]:
# Join the names of population pairs that identify the same window as a robust outlier.
def join_pair_labels(series):
    return "; ".join(sorted(set(series.astype(str))))

recurrence_rows = []

group_cols = [
    "Contig", "Region",
    "Genomic_start_POS", "Genomic_end_POS",
    "Site_class"
]

# Summarise how often each window is identified as a robust outlier
for keys, q in secondary_windows.groupby(group_cols):
    contig, region, start, end, site_class = keys

    ng = q[q["Nigeria_status"].eq("With Nigeria_Gombe")]
    non = q[q["Nigeria_status"].eq("Without Nigeria")]

    recurrence_rows.append({
        "Contig": contig,
        "Region": region,
        "Genomic_start_POS": start,
        "Genomic_end_POS": end,
        "Site_class": site_class,
        "N_Nigeria_pairs_robust": int(ng["Robust_outlier"].sum()),
        "N_nonNigeria_pairs_robust": int(non["Robust_outlier"].sum()),
        "N_all_pairs_robust": int(q["Robust_outlier"].sum()),
        "Max_absolute_FST": q["FST"].max(),
        "Mean_absolute_FST": q["FST"].mean(),
        "Max_FST_excess": q["FST_excess_over_median"].max(),
        "Any_low_site_outlier": bool(
            (q["Robust_outlier"] & q["Low_site_count"]).any()
        ),
        "Outlier_pairs": join_pair_labels(
            q.loc[q["Robust_outlier"], "Pair_label"]
        ) if q["Robust_outlier"].any() else "",
    })

secondary_recurrence = pd.DataFrame(recurrence_rows)

secondary_recurrence["Nigeria_recurrent_2of3_control0"] = (
    (secondary_recurrence["N_Nigeria_pairs_robust"] >= 2)
    & (secondary_recurrence["N_nonNigeria_pairs_robust"] == 0)
)

secondary_recurrence["Nigeria_recurrent_3of3_control0"] = (
    (secondary_recurrence["N_Nigeria_pairs_robust"] == 3)
    & (secondary_recurrence["N_nonNigeria_pairs_robust"] == 0)
)

secondary_recurrence.to_csv(
    TABLE_DIR / "Secondary_window_recurrence_v3_6.csv",
    index=False
)

display(
    secondary_recurrence[
        secondary_recurrence["N_all_pairs_robust"] > 0
    ].sort_values(
        ["Max_absolute_FST", "N_all_pairs_robust"],
        ascending=False
    ).head(30)
)

## Secondary within-pair clusters

In [ ]:
cluster_rows = []
# Analyse robust outlier windows separately for each pair * chromosome * site-class combination.
for (
    pair_id,
    pair_label,
    nigeria_status,
    contig,
    site_class
), q in secondary_windows[
    secondary_windows["Robust_outlier"]
].groupby([
    "Pair_ID", "Pair_label", "Nigeria_status",
    "Contig", "Site_class"
]):
    # Sort outlier windows by genomic position.
    q = q.sort_values("Genomic_start_POS").copy()

    cluster_id = 0
    current = []
    # Walk through the outlier windows and group adjacent windows together.
    for _, row in q.iterrows():
        if not current:
            current = [row]
            continue

        prev = current[-1]

        # Consecutive 1 Mb windows share a boundary plus one base.
        adjacent = (
            int(row["Genomic_start_POS"])
            <= int(prev["Genomic_end_POS"]) + 1_000_001
        )

        if adjacent:
            current.append(row)
        else:
            if len(current) >= 2:
                cluster_id += 1
                cluster_rows.append({
                    "Pair_ID": pair_id,
                    "Pair_label": pair_label,
                    "Nigeria_status": nigeria_status,
                    "Contig": contig,
                    "Site_class": site_class,
                    "Cluster_ID": cluster_id,
                    "N_windows": len(current),
                    "Cluster_start_POS": min(r["Genomic_start_POS"] for r in current),
                    "Cluster_end_POS": max(r["Genomic_end_POS"] for r in current),
                    "Max_FST": max(r["FST"] for r in current),
                    "Max_FST_excess": max(
                        r["FST_excess_over_median"] for r in current
                    ),
                })
            current = [row]
    # Save the current cluster only if it contains at least two adjacent outlier windows.
    if len(current) >= 2:
        cluster_id += 1
        cluster_rows.append({
            "Pair_ID": pair_id,
            "Pair_label": pair_label,
            "Nigeria_status": nigeria_status,
            "Contig": contig,
            "Site_class": site_class,
            "Cluster_ID": cluster_id,
            "N_windows": len(current),
            "Cluster_start_POS": min(r["Genomic_start_POS"] for r in current),
            "Cluster_end_POS": max(r["Genomic_end_POS"] for r in current),
            "Max_FST": max(r["FST"] for r in current),
            "Max_FST_excess": max(
                r["FST_excess_over_median"] for r in current
            ),
        })

secondary_clusters = pd.DataFrame(cluster_rows)

secondary_clusters.to_csv(
    TABLE_DIR / "Secondary_within_pair_clusters_v3_6.csv",
    index=False
)

print("Within-pair multi-window clusters:", len(secondary_clusters))
display(secondary_clusters.head(30))

## Secondary consensus regions and absolute effect-size ranking

In [ ]:
# First keep all exact windows that were robust outliers at least once.
signal_windows = (
    secondary_recurrence[
        secondary_recurrence["N_all_pairs_robust"] > 0
    ]
    .sort_values(["Contig", "Genomic_start_POS", "Site_class"])
    .copy()
)

consensus_rows = []

# Process X and 3R separately.
for contig, q in signal_windows.groupby("Contig"):
    # Merge adjacent signal windows irrespective of site class.
    # And keep each genomic window only once.
    unique_windows = (
        q[[
            "Region", "Genomic_start_POS", "Genomic_end_POS"
        ]]
        .drop_duplicates()
        .sort_values("Genomic_start_POS")
    )

    regions = []
    current = []
    # Merge adjacent signal windows into broader secondary regions.
    for _, row in unique_windows.iterrows():
        if not current:
            current = [row]
            continue

        prev = current[-1]

        adjacent = (
            int(row["Genomic_start_POS"])
            <= int(prev["Genomic_end_POS"]) + 1_000_001
        )

        if adjacent:
            current.append(row)
        else:
            regions.append(current)
            current = [row]

    if current:
        regions.append(current)

    # Summarise recurrence, effect size and QC for each broader region.
    for idx, region_rows in enumerate(regions, start=1):
        start = min(r["Genomic_start_POS"] for r in region_rows)
        end = max(r["Genomic_end_POS"] for r in region_rows)

        members = q[
            (q["Genomic_start_POS"] >= start)
            & (q["Genomic_end_POS"] <= end)
        ]

        ng_max = members["N_Nigeria_pairs_robust"].max()
        non_max = members["N_nonNigeria_pairs_robust"].max()
        max_fst = members["Max_absolute_FST"].max()
        max_excess = members["Max_FST_excess"].max()
        any_low = members["Any_low_site_outlier"].any()

        if (
            ng_max >= 2
            and non_max == 0
            and not any_low
        ):
            priority = "High-priority secondary signal"
        elif members["N_all_pairs_robust"].max() >= 2:
            priority = "Moderate secondary signal"
        else:
            priority = "Single-comparison secondary signal"
        # Store the summary for this broader secondary region.
        consensus_rows.append({
            "Consensus_secondary_region_ID": f"{contig}_secondary_{idx:02d}",
            "Contig": contig,
            "Region_start_POS": start,
            "Region_end_POS": end,
            "N_1Mb_windows": len(region_rows),
            "Max_Nigeria_recurrence": int(ng_max),
            "Max_nonNigeria_recurrence": int(non_max),
            "Max_absolute_FST": max_fst,
            "Max_FST_excess_over_local_median": max_excess,
            "Any_low_site_outlier": bool(any_low),
            "Secondary_priority": priority,
        })

consensus_secondary_regions = pd.DataFrame(consensus_rows)

if not consensus_secondary_regions.empty:
    consensus_secondary_regions = (
        consensus_secondary_regions
        .sort_values(
            ["Max_absolute_FST", "Max_Nigeria_recurrence"],
            ascending=False
        )
        .reset_index(drop=True)
    )

consensus_secondary_regions.to_csv(
    TABLE_DIR / "Secondary_consensus_regions_effectsize_v3_6.csv",
    index=False
)

display(consensus_secondary_regions.head(30))

## Secondary QC summary

In [ ]:
n_robust_obs = int(secondary_windows["Robust_outlier"].sum())
# Count the number of unique 1 Mb genomic windows
n_unique_windows = (
    secondary_windows.loc[
        secondary_windows["Robust_outlier"],
        ["Contig", "Region"]
    ]
    .drop_duplicates()
    .shape[0]
)
# Count window-site signals that recur in at least two of the three Nigeria-containing comparisons
n_nigeria_recurrent_windows = int(
    secondary_recurrence["Nigeria_recurrent_2of3_control0"].sum()
)
# Three of the three
n_nigeria_3of3_windows = int(
    secondary_recurrence["Nigeria_recurrent_3of3_control0"].sum()
)

robust_with_counts = secondary_windows[
    secondary_windows["Robust_outlier"]
    & secondary_windows["N_sites"].notna()
]

n_robust_low_site = int(
    robust_with_counts["Low_site_count"].sum()
)
# Collect the main secondary-landscape results
summary = pd.DataFrame([{
    "Robust_outlier_observations": n_robust_obs,
    "Unique_1Mb_windows_with_outlier": n_unique_windows,
    "Within_pair_multiwindow_clusters": len(secondary_clusters),
    "Consensus_secondary_regions": len(consensus_secondary_regions),
    "Nigeria_recurrent_2of3_control0_window_site_signals":
        n_nigeria_recurrent_windows,
    "Nigeria_recurrent_3of3_control0_window_site_signals":
        n_nigeria_3of3_windows,
    "Robust_outliers_with_verified_site_counts":
        len(robust_with_counts),
    "Robust_outliers_lt1000_sites":
        n_robust_low_site,
}])

summary.to_csv(
    TABLE_DIR / "Secondary_landscape_summary_v3_6.csv",
    index=False
)

display(summary)

## Final structural validation

In [ ]:
# Six pairs: 3 Nigeria-containing + 3 non-Nigeria.
pair_status_check = (
    fst_wide[["Pair_ID", "Nigeria_status"]]
    .drop_duplicates()
    .groupby("Nigeria_status")["Pair_ID"]
    .nunique()
    .to_dict()
)

if pair_status_check != {
    "With Nigeria_Gombe": 3,
    "Without Nigeria": 3
}:
    raise ValueError(
        f"Nigeria classification failed: {pair_status_check}"
    )

# Dataset A = exactly four X windows in every pair.
a_counts = dataset_A.groupby("Pair_ID")["Region"].nunique()

if len(a_counts) != 6 or not (a_counts == 4).all():
    raise ValueError(
        "Dataset A completeness failed:\n"
        + str(a_counts)
    )

# Dataset B contains no focal X windows.
if dataset_B[
    dataset_B["Contig"].eq("X")
    & dataset_B["Region"].astype(str).isin(dataset_A_regions)
].shape[0] != 0:
    raise ValueError(
        "Dataset B still contains Dataset A X windows."
    )

validation = pd.DataFrame([
    ["Six population pairs", fst_wide["Pair_ID"].nunique() == 6],
    ["Three Nigeria-containing pairs",
     pair_status_check.get("With Nigeria_Gombe", 0) == 3],
    ["Three non-Nigeria pairs",
     pair_status_check.get("Without Nigeria", 0) == 3],
    ["Dataset A has four windows in each pair",
     bool((a_counts == 4).all())],
    ["Dataset B excludes Dataset A from X", True],
    ["3R is not focal-region filtered",
     dataset_B[dataset_B["Contig"].eq("3R")]["Region"].nunique()
     == fst_wide[fst_wide["Contig"].eq("3R")]["Region"].nunique()],
    ["Verified site-count columns present",
     {"N_sites_0fold", "N_sites_4fold"}.issubset(fst_wide.columns)],
])

validation.columns = ["Check", "Pass"]
validation.to_csv(
    TABLE_DIR / "FINAL_structural_validation.csv",
    index=False
)

display(validation)

if not validation["Pass"].all():
    raise ValueError("At least one final validation check failed.")

## Optional Dataset A gene table

The original v3.6 notebook obtained gene annotation through the MalariaGEN API. To keep this submission notebook independent of the current `malariagen_data`/Python version, gene annotation is not downloaded here.

If `Table_all_genes_in_Nigeria_X_peak.csv` is archived in `data/final/`, the following cell loads it and checks that the expected focal candidates can be found. This keeps gene annotation as a downstream positional interpretation rather than using gene identity to define the peak.

In [ ]:
# Define path to the candidate gene annotation CSV file
GENE_FILE = DATA_DIR / "Table_all_genes_in_Nigeria_X_peak.csv"

if GENE_FILE.exists():
    # Read candidate gene metadata into DataFrame
    peak_genes = pd.read_csv(GENE_FILE)

    print("Archived focal gene rows:", len(peak_genes))
    # Scan all dataframe columns to verify presence of specific target genes (CYP9K1, CPR125)
    for gene_name in ["CYP9K1", "CPR125"]:
        hit = peak_genes.astype(str).apply(
            lambda col: col.str.contains(
                gene_name,
                case=False,
                na=False
            )
        ).any(axis=1)

        print(gene_name, "rows:", int(hit.sum()))

    display(peak_genes.head(30))

else:
    print(
        "Gene table not found in data/final/. "
        "The FST/statistical analysis is complete without this optional file."
    )

## Final output manifest

In [ ]:
manifest_rows = []

# Recursively iterate through all files and folders in the output directory
for path in sorted(OUTPUT_ROOT.rglob("*")):
    if path.is_file():
        manifest_rows.append({
            "Relative_path": str(path.relative_to(OUTPUT_ROOT)),
            "Size_bytes": path.stat().st_size,
        })

final_manifest = pd.DataFrame(manifest_rows)

final_manifest.to_csv(
    OUTPUT_ROOT / "output_manifest_all_files_FINAL.csv",
    index=False
)

print("Final output files:", len(final_manifest))
display(final_manifest)